# Gravity Forward Modeling Tutorial

This notebook demonstrates a complete gravity forward modeling workflow using the `forward-model` library:
1. Define a geological model
2. Compute the gravity anomaly
3. Plot the results
4. Export data

**Prerequisites:** The observed data you compare against must be the Complete Bouguer Anomaly (CBA). See the [theory docs](../docs/theory.md) for data reduction details.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from forward_model import (
    GeologicBody,
    GravityModel,
    GravityProperties,
    calculate_anomaly,
    plot_combined,
)
from forward_model.io import write_csv

## Step 1: Define the geological model

In [ ]:
# A mafic intrusion: 400 kg/m³ denser than sedimentary host
# Depth: 200–800 m, Width: 400 m, centred on the profile
intrusion = GeologicBody(
    name="Mafic Intrusion",
    gravity=GravityProperties(density_contrast=400.0),
    vertices=[
        [-200.0, 200.0],
        [ 200.0, 200.0],
        [ 200.0, 800.0],
        [-200.0, 800.0],
    ],
)

observation_x = np.linspace(-2000, 2000, 81).tolist()
model = GravityModel(
    bodies=[intrusion],
    observation_x=observation_x,
    observation_z=0.0,
)
print(f"Model has {len(model.bodies)} body, {len(model.observation_x)} observation points")

## Step 2: Compute the gravity anomaly

In [ ]:
result = calculate_anomaly(model)
print(f"gz range: {result.gz.min():.3f} to {result.gz.max():.3f} mGal")
print(f"Peak anomaly at x = {observation_x[result.gz.argmax()]:.0f} m")

## Step 3: Plot the results

In [ ]:
fig = plot_combined(
    model,
    result.gz,
    component="gz",
    gradient=result.gz_gradient,
    style="default",
)
plt.show()

## Step 4: Export results

In [ ]:
write_csv(
    "gravity_results.csv",
    model.observation_x,
    result.gz,
    anomaly_label="anomaly_mGal",
)
print("Saved to gravity_results.csv")

## Next steps

- Try modifying the density contrast or body geometry to see how the anomaly changes
- Explore 2.5D and 2.75D finite-strike extensions using `strike_half_length` or `strike_forward`/`strike_backward`
- Run gravity models from the CLI: `forward-model run examples/dense_intrusion.json`
- See the [theory docs](../docs/theory.md) for the mathematical background